In [1]:

from dataclasses import dataclass
import os

@dataclass(frozen=True)
class Provider:
    """Configuration for routing requests across inference providers."""
    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        is_free=True,
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b"
    )
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    raise RuntimeError(
        "No provider API key found. Please configure GROQ_API_KEY."
    )


In [8]:

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()


def build_client():
    provider = select_provider()

    api_key = os.getenv(provider.env_var)

    return Groq(api_key=api_key)


def llm_reply(prompt: str) -> str:
    provider = select_provider()
    client = build_client()

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return result.choices[0].message.content





In [9]:
print(llm_reply("Who are the cricketers with the most international runs?"))